<a href="https://colab.research.google.com/github/mmarguerittee/ML_projects/blob/main/REAL_ESTATE_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import numpy as np

In [ ]:
import os
os.getcwd()

'/content'

In [ ]:
test=pd.read_excel('test.xlsx')
train=pd.read_excel('train.xlsx')

In [ ]:
def to_rub(çurrency):
  if çurrency=='RUB':
    return 1
  elif çurrency=='USD':
    return 75.73
  else:
    return 94.08


In [ ]:
train['Стоимость']=train['Валюта'].apply(to_rub)*train['Стоимость']

In [ ]:
train=train.drop('Валюта',axis=1)
test=test.drop('Валюта',axis=1)

In [ ]:
import re

In [ ]:
def extract_number(room_desc):
    numbers = re.findall(r'\d+', room_desc)
    if numbers:
        return sum(int(num) for num in numbers)
    else:
        return 1

In [ ]:
train['Количество комнат']=train['Количество комнат'].astype(str)
test['Количество комнат']=test['Количество комнат'].astype(str)

In [ ]:
train['Количество комнат']=train['Количество комнат'].apply(extract_number)
test['Количество комнат']=test['Количество комнат'].apply(extract_number)

In [ ]:
import re

def split1(df, column_name):
    df = df.copy()

    def extract_count(text, type):
        if pd.isna(text):
            return 0
        match = re.search(rf'{type} \((\d+)\)', text)
        return int(match.group(1)) if match else 0

    df['Балконы'] = df[column_name].apply(lambda x: extract_count(x, 'Балкон'))
    df['Лоджии'] = df[column_name].apply(lambda x: extract_count(x, 'Лоджия'))

    return df

In [ ]:
train = split1(train, 'Балкон')
test = split1(test, 'Балкон')

In [ ]:
import re

def split2(df, column_name):
    df = df.copy()

    def extract_count(text, type):
        if pd.isna(text):
            return 0
        match = re.search(rf'{type} \((\d+)\)', text)
        return int(match.group(1)) if match else 0

    df['Санузел совмещенный'] = df[column_name].apply(lambda x: extract_count(x, 'Совмещенный'))
    df['Санузел раздельный'] = df[column_name].apply(lambda x: extract_count(x, 'Раздельный'))

    return df

In [ ]:
train = split2(train, 'Санузел')
test = split2(test, 'Санузел')

In [ ]:
s = train["Метро"].astype(str).str.strip()

train["Метро_станция"] = (
    s.str.replace(r"^\s*м\.\s*", "", regex=True)
     .str.replace(r"\s*\(.*\)\s*$", "", regex=True)
     .replace({"nan": np.nan})
)

m = s.str.extract(r"\((\d+)\s*мин\s*(пешком|на машине)\)", expand=True)
mins = pd.to_numeric(m[0], errors="coerce")
mode = m[1]

train["Метро_мин_пешком"] = np.where(
    mode.eq("на машине"),
    mins * 2,
    mins
).astype("float")

In [ ]:
s = test["Метро"].astype(str).str.strip()

test["Метро_станция"] = (
    s.str.replace(r"^\s*м\.\s*", "", regex=True)
     .str.replace(r"\s*\(.*\)\s*$", "", regex=True)
     .replace({"nan": np.nan})
)

m = s.str.extract(r"\((\d+)\s*мин\s*(пешком|на машине)\)", expand=True)
mins = pd.to_numeric(m[0], errors="coerce")
mode = m[1]

test["Метро_мин_пешком"] = np.where(
    mode.eq("на машине"),
    mins * 2,
    mins
).astype("float")

In [ ]:
import numpy as np
import pandas as pd

col = "Площадь, м2"

s = train[col].astype(str).str.replace(",", ".", regex=False).str.strip()


parts = s.str.split("/", expand=True)

train["Площадь_общая"] = pd.to_numeric(parts[0], errors="coerce")
train["Площадь_жилая"] = pd.to_numeric(parts[1], errors="coerce") if parts.shape[1] > 1 else np.nan
train["Площадь_кухня"] = pd.to_numeric(parts[2], errors="coerce") if parts.shape[1] > 2 else np.nan

living_share = (train["Площадь_жилая"] / train["Площадь_общая"]).mean(skipna=True)
kitchen_share = (train["Площадь_кухня"] / train["Площадь_общая"]).mean(skipna=True)

mask_total = train["Площадь_общая"].notna()

train.loc[mask_total & train["Площадь_жилая"].isna(), "Площадь_жилая"] = (
    train.loc[mask_total & train["Площадь_жилая"].isna(), "Площадь_общая"] * living_share
)

train.loc[mask_total & train["Площадь_кухня"].isna(), "Площадь_кухня"] = (
    train.loc[mask_total & train["Площадь_кухня"].isna(), "Площадь_общая"] * kitchen_share
)

train[["Площадь_общая","Площадь_жилая","Площадь_кухня"]] = train[["Площадь_общая","Площадь_жилая","Площадь_кухня"]].round(1)


In [ ]:
s = test[col].astype(str).str.replace(",", ".", regex=False).str.strip()
test["Площадь_общая"] = pd.to_numeric(parts[0], errors="coerce")
test["Площадь_жилая"] = pd.to_numeric(parts[1], errors="coerce") if parts.shape[1] > 1 else np.nan
test["Площадь_кухня"] = pd.to_numeric(parts[2], errors="coerce") if parts.shape[1] > 2 else np.nan
mask_total = test["Площадь_общая"].notna()

test.loc[mask_total & test["Площадь_жилая"].isna(), "Площадь_жилая"] = (
    test.loc[mask_total & test["Площадь_жилая"].isna(), "Площадь_общая"] * living_share
)

test.loc[mask_total & test["Площадь_кухня"].isna(), "Площадь_кухня"] = (
    test.loc[mask_total & test["Площадь_кухня"].isna(), "Площадь_общая"] * kitchen_share
)

test[["Площадь_общая","Площадь_жилая","Площадь_кухня"]] = test[["Площадь_общая","Площадь_жилая","Площадь_кухня"]].round(1)

In [ ]:
s = train["Дом"].astype(str).str.strip()

floors = s.str.extract(r"^\s*(\d+)\s*/\s*(\d+)", expand=True)
train["Этаж"] = pd.to_numeric(floors[0], errors="coerce")
train["Этажность"] = pd.to_numeric(floors[1], errors="coerce")

mat = s.str.extract(r",\s*([^,]+)\s*$", expand=True)[0]
train["Материал"] = mat.where(mat.notna() & (mat != ""), np.nan)

train["Материал"] = train["Материал"].astype("string").str.strip()

In [ ]:
s = test["Дом"].astype(str).str.strip()

floors = s.str.extract(r"^\s*(\d+)\s*/\s*(\d+)", expand=True)
test["Этаж"] = pd.to_numeric(floors[0], errors="coerce")
test["Этажность"] = pd.to_numeric(floors[1], errors="coerce")

mat = s.str.extract(r",\s*([^,]+)\s*$", expand=True)[0]
test["Материал"] = mat.where(mat.notna() & (mat != ""), np.nan)

test["Материал"] = test["Материал"].astype("string").str.strip()

In [ ]:
train['Ремонт']=train['Ремонт'].fillna('Без ремонта')
test['Ремонт']=test['Ремонт'].fillna('Без ремонта')

In [ ]:
s = train["Адрес"].astype(str)
train["Город"] = (
    s.str.extract(r"(Москва|Санкт-Петербург|Екатеринбург)", expand=False)
     .astype("string")
)

In [ ]:
s = test["Адрес"].astype(str)
test["Город"] = (
    s.str.extract(r"(Москва|Санкт-Петербург|Екатеринбург)", expand=False)
     .astype("string")
)

In [ ]:
import re

def split(df, column_name):
    df = df.copy()

    def extract_count(text, type):
        if pd.isna(text):
            return 0
        match = re.search(rf'{type} \((\d+)\)', text)
        return int(match.group(1)) if match else 0

    df['Пассажирский_лифт'] = df[column_name].apply(lambda x: extract_count(x, 'Пасс'))
    df['Грузовой_лифт'] = df[column_name].apply(lambda x: extract_count(x, 'Груз'))

    return df

In [ ]:
train = split(train, 'Лифт')
test = split(test, 'Лифт')

In [ ]:
def extract_year_from_text(text):
    if pd.isna(text):
        return None

    text = str(text)
    match = re.search(r'(20\d{2}|19\d{2})', text)
    if match:
        return int(match.group(1))
    return None

train['Год_постройки'] = train['Название ЖК'].apply(extract_year_from_text)
test['Год_постройки'] = test['Название ЖК'].apply(extract_year_from_text)

year_mode = train['Год_постройки'].mode()[0]
train['Год_постройки'] = train['Год_постройки'].fillna(year_mode)
test['Год_постройки'] = test['Год_постройки'].fillna(year_mode)

from datetime import datetime
current_year = datetime.now().year
train['Возраст_дома'] = current_year - train['Год_постройки']
test['Возраст_дома'] = current_year - test['Год_постройки']

In [ ]:
train['Тип'].value_counts()

,count
Тип,
Продажа квартиры,2261
Продажа квартиры в новостройке,260
Продажа доли в квартире,11


In [ ]:
df1=train[['Год_постройки','Возраст_дома','Балконы', "Лоджии", 'Санузел совмещенный', "Санузел раздельный", "Мусоропровод", 'Пассажирский_лифт', 'Грузовой_лифт','Парковка','Высота потолков, м','Количество комнат','Тип','Метро_станция','Метро_мин_пешком',"Площадь_общая","Площадь_жилая","Площадь_кухня","Этаж","Этажность","Материал",'Ремонт','Город']]
df2=test[['Год_постройки','Возраст_дома','Балконы', "Лоджии", 'Санузел совмещенный', "Санузел раздельный", "Мусоропровод", 'Пассажирский_лифт', 'Грузовой_лифт','Парковка','Высота потолков, м','Количество комнат','Тип','Метро_станция','Метро_мин_пешком',"Площадь_общая","Площадь_жилая","Площадь_кухня","Этаж","Этажность","Материал",'Ремонт','Город']]

In [ ]:
y_train=train['Стоимость']

In [ ]:
def trash(df, column_name):
    df = df.copy()
    df[column_name] = (
        df[column_name]
        .fillna('нет')
        .str.strip()
        .str.lower()
        .eq('да')
        .astype(int)
    )
    return df
df1 = trash(df1, 'Мусоропровод')
df2 = trash(df2, 'Мусоропровод')

In [ ]:
df1['Парковка']=df1['Парковка'].fillna('нет')
df2['Парковка']=df2['Парковка'].fillna('нет')

In [ ]:
df1['Высота потолков, м']=df1['Высота потолков, м'].fillna(2.9)
df2['Высота потолков, м']=df2['Высота потолков, м'].fillna(2.9)

In [ ]:
df1[df1['Город']=='Москва']['Метро_станция'].mode()


,Метро_станция
0,Парк Победы


In [ ]:
df1[df1['Город']=='Санкт-Петербург']['Метро_станция'].mode()


,Метро_станция
0,Комендантский проспект


In [ ]:
df1[df1['Город']=='Екатеринбург']['Метро_станция'].mode()

,Метро_станция
0,Ботаническая


In [ ]:
station = {
    'Москва': 'Парк Победы',
    'Санкт-Петербург': 'Комендантский проспект',
    'Екатеринбург': 'Ботаническая'
}

df1['Метро_станция'] = df1['Метро_станция'].fillna(
    df1['Город'].map(station)
)

df2['Метро_станция'] = df2['Метро_станция'].fillna(
    df2['Город'].map(station)
)

In [ ]:
df1['Метро_мин_пешком'].mean()

np.float64(15.070775347912525)

In [ ]:
df1['Метро_мин_пешком'] = df1['Метро_мин_пешком'].fillna(15)
df2['Метро_мин_пешком'] = df2['Метро_мин_пешком'].fillna(15)

In [ ]:
df1['Материал'].mode()

,Материал
0,Монолитный


In [ ]:
df1['Материал']=df1['Материал'].fillna('Монолитный')
df2['Материал']=df2['Материал'].fillna('Монолитный')

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2532 entries, 0 to 2531
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Год_постройки        2532 non-null   float64
 1   Возраст_дома         2532 non-null   float64
 2   Балконы              2532 non-null   int64  
 3   Лоджии               2532 non-null   int64  
 4   Санузел совмещенный  2532 non-null   int64  
 5   Санузел раздельный   2532 non-null   int64  
 6   Мусоропровод         2532 non-null   int64  
 7   Пассажирский_лифт    2532 non-null   int64  
 8   Грузовой_лифт        2532 non-null   int64  
 9   Парковка             2532 non-null   object 
 10  Высота потолков, м   2532 non-null   float64
 11  Количество комнат    2532 non-null   int64  
 12  Тип                  2532 non-null   object 
 13  Метро_станция        2532 non-null   object 
 14  Метро_мин_пешком     2532 non-null   float64
 15  Площадь_общая        2532 non-null   f

In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 633 entries, 0 to 632
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Год_постройки        633 non-null    float64
 1   Возраст_дома         633 non-null    float64
 2   Балконы              633 non-null    int64  
 3   Лоджии               633 non-null    int64  
 4   Санузел совмещенный  633 non-null    int64  
 5   Санузел раздельный   633 non-null    int64  
 6   Мусоропровод         633 non-null    int64  
 7   Пассажирский_лифт    633 non-null    int64  
 8   Грузовой_лифт        633 non-null    int64  
 9   Парковка             633 non-null    object 
 10  Высота потолков, м   633 non-null    float64
 11  Количество комнат    633 non-null    int64  
 12  Тип                  633 non-null    object 
 13  Метро_станция        633 non-null    object 
 14  Метро_мин_пешком     633 non-null    float64
 15  Площадь_общая        633 non-null    flo

KNN (K-Nearest Neighbours)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train = df1.copy()
X_test  = df2.copy()

num_cols = [
     "Количество комнат", "Метро_мин_пешком",
     "Площадь_общая", "Площадь_жилая", "Площадь_кухня",
     "Этаж", "Этажность", 'Высота потолков, м', 'Балконы', "Лоджии",
     'Санузел совмещенный', "Санузел раздельный", "Мусоропровод",
     'Пассажирский_лифт', 'Грузовой_лифт', 'Год_постройки','Возраст_дома'
     ]
cat_cols = ["Тип", "Метро_станция", "Материал", "Ремонт", "Город", 'Парковка']

preprocess = ColumnTransformer(
     transformers=[
         ("num", StandardScaler(), num_cols),
         ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
         ],
     remainder="drop"
 )

pipe = Pipeline(steps=[
     ("prep", preprocess),
     ("knn", KNeighborsRegressor())
 ])

param_grid = {
     "knn__n_neighbors": list(range(3, 21)),
     "knn__weights": ["distance"],
     "knn__p": [1, 2], # 1=Manhattan, 2=Euclidean
 }

gs = GridSearchCV(
     pipe,
     param_grid=param_grid,
     cv=5,
     scoring="neg_mean_absolute_error",
     n_jobs=-1,
     verbose=1
)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best CV MAE:", -gs.best_score_)

y_pred_test = gs.predict(X_test)


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV MAE: 8019143.295553473


In [ ]:
gs.cv_results_

{'mean_fit_time': array([0.01861944, 0.01921811, 0.01899619, 0.0193295 , 0.01970463,
        0.01853733, 0.0185739 , 0.01867352, 0.01776266, 0.01858859,
        0.01848178, 0.01935205, 0.01863918, 0.01799355, 0.01801596,
        0.0187253 , 0.02743964, 0.03680902, 0.0329143 , 0.03536396,
        0.01898293, 0.01915951, 0.01721401, 0.01785126, 0.01922636,
        0.01837335, 0.01768346, 0.01828666, 0.01862731, 0.01780624,
        0.01844215, 0.01867275, 0.0182621 , 0.02297544, 0.01769085,
        0.01813602]),
 'std_fit_time': array([0.00257952, 0.00224388, 0.00112388, 0.00073923, 0.00174208,
        0.00156484, 0.00103102, 0.00112971, 0.00064035, 0.00012306,
        0.00106162, 0.00066136, 0.00120363, 0.00010455, 0.0002657 ,
        0.00052101, 0.00794518, 0.00614059, 0.00383455, 0.00770233,
        0.00308071, 0.00180195, 0.00065471, 0.00165134, 0.00044518,
        0.00073889, 0.00088433, 0.00150835, 0.00106896, 0.00081849,
        0.00034235, 0.00067537, 0.00038855, 0.00847803, 0.001

In [ ]:
subm=pd.DataFrame({'id':[i for i in range(test.shape[0])],'Стоимость':y_pred_test})

In [ ]:
test.shape[0]

633

In [ ]:
subm

,id,Стоимость
0,0,7.922265e+06
1,1,6.166674e+06
2,2,7.629627e+06
3,3,6.607098e+06
4,4,7.087135e+07
...,...,...
628,628,1.201427e+07
629,629,1.072020e+07
630,630,8.663884e+06
631,631,8.974451e+06


In [ ]:
subm.to_csv('my_solution.csv',index=False)
